## Install Dependancies


In [ ]:
# 1. Install flex and bison
!apt-get install -y flex bison

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libfl-dev libfl2
Suggested packages:
  bison-doc flex-doc
The following NEW packages will be installed:
  bison flex libfl-dev libfl2
0 upgraded, 4 newly installed, 0 to remove and 5 not upgraded.
Need to get 1,072 kB of archives.
After this operation, 3,667 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 flex amd64 2.6.4-8build2 [307 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 bison amd64 2:3.8.2+dfsg-1build1 [748 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfl2 amd64 2.6.4-8build2 [10.7 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/main amd64 libfl-dev amd64 2.6.4-8build2 [6,236 B]
Fetched 1,072 kB in 0s (8,168 kB/s)
Selecting previously unselected package flex.
(Reading database ... 118194 files and directories currently installed.)
Preparing to un

In [ ]:
%%writefile compi.l
%{
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include "compi.tab.h"

int line = 1;
%}

%x MULTILINE_COMMENT

DIGIT      [0-9]
LETTER     [A-Za-z_]
ID         {LETTER}({LETTER}|{DIGIT})*
ICONST     0|[1-9]{DIGIT}*
FCONST     {DIGIT}+"."{DIGIT}+

%%

"//".* { /* skip */ }
"/*"                    { BEGIN(MULTILINE_COMMENT); }
<MULTILINE_COMMENT>"*/" { BEGIN(INITIAL); }
<MULTILINE_COMMENT>\n   { line++; }
<MULTILINE_COMMENT>.    { /* skip */ }

"int"        { return INT; }
"float"      { return FLOAT; }
"if"         { return IF; }
"else"       { return ELSE; }
"while"      { return WHILE; }
"print"      { return PRINT; }

{FCONST}     { yylval.fval = atof(yytext); return FCONST; }
{ICONST}     { yylval.ival = atoi(yytext); return ICONST; }
{ID}         { yylval.sval = strdup(yytext); return ID; }

"=="         { return EQ; }
"!="         { return NE; }
"<="         { return LE; }
">="         { return GE; }
"&&"         { return AND; }
"||"         { return OR; }

"<"          { return '<'; }
">"          { return '>'; }
"="          { return '='; }
"+"          { return '+'; }
"-"          { return '-'; }
"*"          { return '*'; }
"/"          { return '/'; }
"%"          { return '%'; }
"!"          { return '!'; }
"("          { return '('; }
")"          { return ')'; }
"{"          { return '{'; }
"}"          { return '}'; }
";"          { return ';'; }
","          { return ','; }

[ \t\r]+     { /* skip whitespace */ }
\n           { line++; }

.            { printf("Lexical error at line %d: Unexpected character '%s'\n", line, yytext); }

%%

int yywrap() { return 1; }

Overwriting compi.l


In [ ]:
%%writefile compi.y
%{
#include <stdio.h>
#include <stdlib.h>

extern int line;
int yylex(void);
void yyerror(const char *s);
%}

%define parse.error verbose

%token-table

%union {
    int ival;
    float fval;
    char* sval;
}

%token <sval> ID
%token <ival> ICONST
%token <fval> FCONST
%token INT FLOAT IF ELSE WHILE PRINT
%token EQ NE LE GE AND OR

/* Operator Precedence and Associativity */
%left OR
%left AND
%left EQ NE LE GE '<' '>'
%left '+' '-'
%left '*' '/'
%left '%'
%right '!'
%nonassoc LOWER_THAN_ELSE
%nonassoc ELSE

%%
program:
    unit_list
    ;

unit_list:
    unit_list unit
    | /* empty */
    ;

unit:
    declaration
    | statement
    | error ';' {
        yyerrok;
        yyclearin;
        printf("Line %d: Recovering from syntax error, skipping to next semicolon.\n", line);
    }
    ;

declaration:
    type ID ';' { printf("Line %d: Syntactic Validation [Declaration: %s]\n", line, $2); free($2); }
    ;

type:
    INT | FLOAT
    ;

statement:
    assignment_stmt
    | if_stmt
    | while_stmt
    | print_stmt
    | compound_stmt
    ;

/* Allows declarations and statements to mix inside braces */
compound_stmt:
    '{' unit_list '}'
    ;

assignment_stmt:
    ID '=' expression ';' { printf("Line %d: Syntactic Validation [Assignment to %s]\n", line, $1); free($1); }
    ;

if_stmt:
    IF '(' condition ')' statement %prec LOWER_THAN_ELSE
    | IF '(' condition ')' statement ELSE statement { printf("Line %d: Syntactic Validation [If-Else Block]\n", line); }
    ;

while_stmt:
    WHILE '(' condition ')' statement { printf("Line %d: Syntactic Validation [While Loop]\n", line); }
    ;

print_stmt:
    PRINT '(' expression ')' ';' { printf("Line %d: Syntactic Validation [Print Statement]\n", line); }
    ;

expression:
    expression '+' expression
    | expression '-' expression
    | expression '*' expression
    | expression '/' expression
    | expression '%' expression
    | '(' expression ')'
    | ID { free($1); }
    | ICONST
    | FCONST
    ;

condition:
    expression EQ expression
    | expression NE expression
    | expression '<' expression
    | expression '>' expression
    | expression LE expression
    | expression GE expression
    | '(' condition ')'
    | '!' condition
    | condition AND condition
    | condition OR condition
    ;

%%

int main() {
    if (yyparse() == 0) {
        printf("\nRESULT: Syntactic Validation Successful.\n");
    } else {
        printf("\nRESULT: Syntactic Validation Failed.\n");
    }
    return 0;
}

// Declare yytname, which is generated by Bison when %token-table is used.
extern const char *const yytname[];

void yyerror(const char *s) {
    fprintf(stderr, "Syntax Error at line %d: %s\n", line, s);
}

Overwriting compi.y


In [ ]:
%%writefile program.txt
// --- SECTION 1: DEEP NESTING & SCOPE TEST --- file fromn hell
int a; int b; int c; int d; int e; int f;
a = 1; b = 2; c = 3; d = 4; e = 5; f = 6;

while (a < 100) {
    if (b > a) {
        while (c < 50) {
            if (d == c) {
                while (e != 0) {
                    if (f <= e) {
                        print(f);
                    } else {
                        print(e);
                    }
                    e = e - 1;
                }
            }
            c = c + 1;
        }
    }
    a = a + 1;
}

// --- SECTION 2: BOOLEAN & OPERATOR PRECEDENCE TORTURE ---
// Tests if ! binds tighter than &&, and * tighter than +
if (!(a + b * c > d / e % f) && (a == b || c != d && e >= f)) {
    print(1);
}

// --- SECTION 3: ERROR GAUNTLET (RECOVERY TEST) ---

// ERROR 1: Missing semicolon in middle of chain
a = 10
b = 20;

// ERROR 2: Expression on left side (L-Value error)
a + b = 30;

// ERROR 3: Missing closing parenthesis in complex IF
if ((a > b && (c < d) {
    print(99);
}

// ERROR 4: The "Semicolon Trap"
if (a == 10);
{
    print(10);
}
else {
    print(0);
}

// ERROR 5: Consecutive keywords (should fail)
if while (a) { print(a); }

// --- SECTION 4: TYPES & CONSTANTS ---
float x;
x = 10.555 + 0.444;
int 123invalid; // ERROR 6: ID starting with digit

// --- SECTION 5: MASSIVE COMPOUND BLOCK ---
{
    int i;
    i = 0;
    while(i < 10){
        {
            {
                // Deeply nested empty-ish blocks
                print(i);
            }
        }
        i = i + 1;
    }
}

// --- SECTION 6: LEXICAL NOISE ---
// These should trigger your '.' rule in Lex
@
#
$
^
& // This is NOT '&&', so it should be a lexical error
`

// Final valid statement to see if it recovered
print(a + b + c + d + e + f);

Overwriting program.txt


In [ ]:
# 2. Generate Parser and Lexer files
!bison -d compi.y
!flex compi.l

# 3. Compile the C files
!gcc compi.tab.c lex.yy.c -o compiler -lfl

# 4. Run the program
!./compiler < program.txt

Line 2: Syntactic Validation [Declaration: a]
Line 2: Syntactic Validation [Declaration: b]
Line 2: Syntactic Validation [Declaration: c]
Line 2: Syntactic Validation [Declaration: d]
Line 2: Syntactic Validation [Declaration: e]
Line 2: Syntactic Validation [Declaration: f]
Line 3: Syntactic Validation [Assignment to a]
Line 3: Syntactic Validation [Assignment to b]
Line 3: Syntactic Validation [Assignment to c]
Line 3: Syntactic Validation [Assignment to d]
Line 3: Syntactic Validation [Assignment to e]
Line 3: Syntactic Validation [Assignment to f]
Line 11: Syntactic Validation [Print Statement]
Line 13: Syntactic Validation [Print Statement]
Line 14: Syntactic Validation [If-Else Block]
Line 15: Syntactic Validation [Assignment to e]
Line 16: Syntactic Validation [While Loop]
Line 18: Syntactic Validation [Assignment to c]
Line 19: Syntactic Validation [While Loop]
Line 21: Syntactic Validation [Assignment to a]
Line 22: Syntactic Validation [While Loop]
Line 27: Syntactic Validati